# Model Results Analysis

**20260312** — Load saved models and RESULTS from the model comparison notebook for analysis.

**Prerequisites:** Run the model comparison notebook (`20260312_model_comparison.ipynb`) and execute the "Export models for analysis" cell to save artifacts to `backend/artifacts/models_v2/`.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
for _p in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_p / "src").is_dir():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.model.export_utils import load_model_bundle, predict, get_model
import src.paths as PATHS

In [ ]:
# Paths
ARTIFACTS_DIR = PATHS.ROOT_DIR / "artifacts" / "models_v2"
FEAT_PATH = PATHS.DATA_DIR / "02_processed" / "erosion" / "region_features.parquet"

if not ARTIFACTS_DIR.exists():
    raise FileNotFoundError(
        f"Artifacts not found at {ARTIFACTS_DIR}. "
        "Run the model comparison notebook and execute the 'Export models' cell first."
    )

bundle = load_model_bundle(ARTIFACTS_DIR)
print(f"Loaded bundle from {ARTIFACTS_DIR}")
print(f"Config keys: {list(bundle['config'].keys())}")
print(f"Results: {list(bundle['results'].keys())}")

## Results summary

In [ ]:
results = bundle["results"]
summary = pd.DataFrame(results).T

# Reorder columns for readability
cols = ["test_mae", "test_rmse", "test_tail_mae", "test_r2", "train_mae", "train_rmse", "train_r2"]
cols = [c for c in cols if c in summary.columns]
summary = summary[cols]

display(summary.round(4))

## Run predictions on new data

In [ ]:
# Load data (same as training notebook)
feat = pd.read_parquet(FEAT_PATH)
test = feat[feat["split"] == "test"].copy()
TARGET = bundle["config"]["TARGET"]

# Get predictions from each model
test = test.copy()
test["pred_ols"] = predict(bundle, test, "ols")
test["pred_ridge_num"] = predict(bundle, test, "ridge_num")
test["pred_ridge_cat"] = predict(bundle, test, "ridge_cat")
test["pred_lgb"] = predict(bundle, test, "lgb")

test[[TARGET, "pred_ols", "pred_ridge_num", "pred_ridge_cat", "pred_lgb"]].head(10)

## Inspect model internals (e.g. OLS coefficients, Ridge weights)

In [ ]:
# OLS: single coefficient
ols = get_model(bundle, "ols")
feats_ols = bundle["config"]["FEATS_2"]
print("OLS coefficients:")
for f, c in zip(feats_ols, ols.coef_):
    print(f"  {f}: {c:.4f}")
print(f"  intercept: {ols.intercept_:.4f}")

# Ridge + categorical: coefficients
ridge_cat = get_model(bundle, "ridge_cat")
feats_ridge = bundle["config"]["FEATS_4"]
coefs = ridge_cat.named_steps["ridge"].coef_
coef_series = pd.Series(coefs, index=feats_ridge).sort_values()
print("\nRidge + categorical coefficients (sorted):")
display(coef_series)